# Ripple detection


##### Imports

In [ ]:
import os
import yaml

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from probeinterface import read_probeinterface
from probeinterface.plotting import plot_probe
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

##### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

raw_data_dir_env = "RAW_DATA_DIR_DUAL"
data_fname_env = "SLEEP_TEST_DATA_FNAME_HPC"
probe_config_env = "PROBE_CONFIG_PATH_HPC"
derivatives_dir_env = "DERIVATIVES_DIR_DUAL"

raw_data_dir = Path(os.getenv(raw_data_dir_env))
data_fname = Path(os.getenv(data_fname_env))
probe_config_path = Path(os.getenv(probe_config_env))
derivatives_dir = Path(os.getenv(derivatives_dir_env))
full_data_path = os.path.join(raw_data_dir, data_fname)
config_path = os.path.join(repo_root, "config/preprocessing.yml")
lfp_output_dir = os.path.join(derivatives_dir, "lfp")

# Load preprocessing config
with open(Path(config_path)) as f:
    config = yaml.safe_load(f)
    
sample_rate = config['recording']['sample_rate']
n_channels = config['recording']['n_channels']
dtype = config['recording']['dtype']
freq_min = config['lfp']['bandpass_filter']['freq_min']
freq_max = config['lfp']['bandpass_filter']['freq_max']
freq_min_highpass = config['preprocessing']['bandpass_filter']['freq_min']
freq_max_highpass = config['preprocessing']['bandpass_filter']['freq_max']
reference = config['lfp']['common_reference']['reference']
operator = config['lfp']['common_reference']['operator']
resample_rate = config['lfp']['resample_rate']

In [ ]:
recording = se.read_binary(full_data_path, sample_rate, dtype, n_channels)

##### Read probe layout from config file

In [ ]:
probe_group = read_probeinterface(probe_config_path)
probe = probe_group.probes[0]

active_mask = probe.device_channel_indices != -1
active_probe = probe.get_slice(active_mask)

# active_probe has 384 contacts, all mapped 1:1 to recording channels.
# The full probe has 5120 contacts (384 active); copy_metadata in preprocessors silently
# drops it when contact count != channel count, causing the "no Probe attached" warning.
recording_with_probe = recording.set_probe(active_probe)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 12), dpi=200)
colors = ["blue" if ch != -1 else "white" for ch in probe.device_channel_indices]
plot_probe(probe, ax=ax, contacts_colors=colors, with_device_index=False)
plt.tight_layout()

##### LFP Preprocessing

In [ ]:
import warnings

# Preprocessing pipeline (lazy chain - no data is loaded yet)
recording_filtered = sp.bandpass_filter(recording_with_probe, freq_min=freq_min, freq_max=freq_max)
recording_cmr = sp.common_reference(recording_filtered, reference=reference, operator=operator)
recording_lfp = sp.resample(recording_cmr, resample_rate=resample_rate)

# Save to disk (skip if folder already exists)
lfp_save_path = Path(lfp_output_dir) / 'lfp_preprocessed' / data_fname.stem
if not lfp_save_path.exists():
    print('Saving preprocessed LFP to disk...')
    # Worker processes reconstruct the chain via dump_to_dict, which doesn't carry probe
    # metadata from in-place set_probe calls. This doesn't affect computation: global
    # median CMR doesn't use electrode positions. Suppress the cosmetic warning.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="There is no Probe attached", category=UserWarning)
        recording_lfp.save(folder=lfp_save_path, n_jobs=20, chunk_duration='1s')
    print(f'Saved to {lfp_save_path}')
else:
    print(f'Found existing LFP recording: {lfp_save_path}')

##### ... or load from saved file

In [ ]:
lfp_save_path = Path(lfp_output_dir) / 'lfp_preprocessed' / data_fname.stem
recording_lfp = si.load(lfp_save_path)
print(recording_lfp)

##### Compute PSD for each channel

In [ ]:
from scipy.signal import welch

fs = recording_lfp.get_sampling_frequency()
traces = recording_lfp.get_traces().astype(np.float32)  # (n_samples, n_channels)

# Welch PSD per channel; nperseg=2*fs gives 0.5 Hz resolution
freqs, psd = welch(traces, fs=fs, nperseg=int(2 * fs), axis=0)  # psd: (n_freqs, n_channels)

In [ ]:
ripple_band = (100, 200)
mask = (freqs >= ripple_band[0]) & (freqs <= ripple_band[1])
band_power = np.trapezoid(psd[mask], freqs[mask], axis=0)  # (n_channels,)

order = np.argsort(band_power)[::-1]
channel_ids = recording_lfp.channel_ids
print(f"Top 10 channels by {ripple_band[0]}-{ripple_band[1]} Hz power:")
for ch in order[:10]:
    print(f"  {channel_ids[ch]}: {band_power[ch]:.3g}")

In [ ]:
locs = recording_with_probe.get_channel_locations()  # same channel order as recording_lfp
xs, depths = locs[:, 0], locs[:, 1]  # um
shank = np.round((xs - xs.min()) / 250).astype(int)  # NeuropixelsV2 shank pitch = 250 um

fig, ax = plt.subplots(figsize=(4, 10))
for s in np.unique(shank):
    sel = shank == s
    ax.scatter(band_power[sel], depths[sel], s=12, label=f"shank {s}")
ax.set_xlabel(f"{ripple_band[0]}-{ripple_band[1]} Hz power")
ax.set_ylabel("Depth (um)")
ax.set_title("Ripple-band power by depth")
ax.legend()
plt.tight_layout()

In [ ]:
best_per_shank = {}
for s in np.unique(shank):
    sel = np.where(shank == s)[0]
    best_per_shank[s] = sel[np.argmax(band_power[sel])]

fmask = (freqs >= 0.25) & (freqs <= 300)
fig, ax = plt.subplots(figsize=(8, 5))
for s, ch in best_per_shank.items():
    ax.semilogy(freqs[fmask], psd[fmask, ch], label=f"shank {s} - {channel_ids[ch]}")
ax.axvspan(ripple_band[0], ripple_band[1], color="grey", alpha=0.2)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD")
ax.set_title("PSD of peak ripple-band channel per shank")
ax.legend()
plt.tight_layout()

In [ ]:
n_per_shank = 4  # peak channel + 3 spanning down to lowest ripple power
shanks = np.unique(shank)
fig, axes = plt.subplots(1, len(shanks), figsize=(5 * len(shanks), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, s in zip(axes, shanks):
    sel = np.where(shank == s)[0]
    ranked = sel[np.argsort(band_power[sel])[::-1]]  # high -> low ripple power
    picks = ranked[np.linspace(0, len(ranked) - 1, n_per_shank).astype(int)]
    for ch in picks:
        ax.semilogy(freqs[fmask], psd[fmask, ch],
                    label=f"{channel_ids[ch]} | {ripple_band[0]}-{ripple_band[1]}Hz pow={band_power[ch]:.3g}")
    ax.axvspan(ripple_band[0], ripple_band[1], color="grey", alpha=0.2)
    ax.set_title(f"shank {s}")
    ax.set_xlabel("Frequency (Hz)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("PSD")
plt.tight_layout()

In [ ]:
best_per_shank

In [ ]:
from scipy.signal import butter, hilbert, sosfilt

trace = traces[:, best_per_shank[0]]
sos = butter(2, ripple_band, btype='bandpass', fs=fs, output='sos')
filtered = sosfilt(sos, trace)
analytic = hilbert(filtered)
envelope = np.abs(analytic)
phase = np.angle(analytic)
inst_freq = np.diff(np.unwrap(phase)) * fs / (2 * np.pi)
start = 0
end = 125000
to_plot = envelope[start:end]
plt.plot(to_plot)

In [ ]:
# Get 4 standard deviations of the ripple-band filtered signal for thresholding
std_dev = np.std(envelope)
threshold = 4 * std_dev
trace_thresholded = np.where(envelope > threshold, envelope, 0)
plt.plot(trace_thresholded[start:end])

In [ ]:
# Get individual number of ripple events detected
# Should be threshold crossings of the ripple-band filtered signal with minimum return to 0 of 100ms between events
# Get continous events
events = []
above_threshold = envelope > threshold
event_start = None
for i in range(len(above_threshold)):
    if above_threshold[i] and event_start is None:
        event_start = i
    elif not above_threshold[i] and event_start is not None:
        event_end = i
        events.append((event_start, event_end))
        event_start = None

# Remove events < 40ms or > 200ms
min_duration = int(0.02 * fs)  # 40 ms
max_duration = int(0.2 * fs)   # 200 ms
filtered_events = [(start, end) for start, end in events if min_duration <= (end - start) <= max_duration]
filtered_events = np.array(filtered_events)
filtered_events

In [ ]:
# Plot a detected ripple event
event_idx = 5
extra_samples = 500
event_start, event_end = filtered_events[event_idx]
event_trace = trace[event_start-extra_samples:event_end+extra_samples]
event_time = np.arange(event_start-extra_samples, event_end+extra_samples) / fs  # Convert to seconds
plt.figure(figsize=(10, 4))
# Add vertical line marking beginning and end of detected ripple event
plt.plot(event_time, event_trace)
plt.vlines((event_time[extra_samples], event_time[-extra_samples]), event_trace.min(), event_trace.max(), colors='red', linestyles='dashed')